# Load Dependencies

In [ ]:
# Data analysis libraries
import numpy as np
import pandas as pd; pd.options.display.max_columns = 200
import geopandas as gpd
import linref as lr
import pyproj

# Visualization libraries
import plotly.express as px
import folium
import branca

# Utility libraries
import os

In [ ]:
# Define global variables
PROJECT_CRS = pyproj.CRS.from_user_input('EPSG:3857')

# Load Data

In [ ]:
# Point this to the location of the Geopackage file
fp = os.path.join('..', '99_Resources', 'franklin_county_training_data.gpkg')

# List all the layers in the file
gpd.list_layers(fp)

In [ ]:
# Load roadway data
segments = gpd.read_file(fp, layer='hin')
segments.to_crs(PROJECT_CRS, inplace=True)

# Load crash data
query = """
SELECT OBJECTID, NLFID, COUNTY_LOG_NBR, CRASH_YR, KABCO, CRASH_TYPE_SIMPLE, DAY_IN_WEEK_TEXT, HOUR_PERIOD, geom
FROM crashes_enriched
"""
crashes = gpd.read_file(fp, sql=query)
crashes.to_crs(PROJECT_CRS, inplace=True)

print(f'Data loaded: {len(segments):,.0f} HIN segments, {len(crashes):,.0f} crashes')

# Create HTML-based Webmaps with `folium`

In [ ]:
# Initialize the map with `folium`
m = folium.Map(tiles=None) # No tiles to start--we'll add these later

# Define the variable parameters to use for desired map layers
HIN_PARAMS = [
    {
        'name': 'KABC Vehicle Score',
        'column': 'KABC_VEH_SCORE',
        'color_map': 'YlOrRd',
        'style_kwds': dict(weight=3),
        'popup': True,
        'tooltip': False
    },
    {
        'name': 'KABC Pedestrian Score',
        'column': 'KABC_PED_SCORE',
        'color_map': 'YlOrRd',
        'style_kwds': dict(weight=3),
        'popup': True,
        'tooltip': False
    }
]

# Add all HIN layers
for PARAMS in HIN_PARAMS:
    MAX_VAL = segments[PARAMS['column']].quantile(0.95)
    segments.explore(
        m=m,
        vmax=MAX_VAL,
        show=False,
        **PARAMS,
    )

In [ ]:
# Format map
folium.TileLayer('OpenStreetMap', name='Street Map', show=True).add_to(m)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    name='Esri Aerial Imagery', attr='Esri', show=False).add_to(m)
folium.TileLayer('cartodbpositron', name='Bright', show=False).add_to(m)
folium.TileLayer('cartodbdark_matter', name='Dark', show=False).add_to(m)
folium.map.LayerControl('bottomright', collapsed=True, autoZIndex=True, draggable=False).add_to(m)
m.fit_bounds(m.get_bounds())


In [ ]:
# Export HTML file
fp = os.path.join('..', '99_Resources', 'hin-webmap-basic.html')
m.save(fp)

# Advanced Mapping
Example: https://geopandas.org/en/stable/gallery/polygon_plotting_with_folium.html

In [ ]:
HIN_PARAMS = [
    {
        'name': 'KABC Vehicle Score',
        'column': 'KABC_VEH',
        'mode': 'Vehicle',
    },
    {
        'name': 'KABC Pedestrian Score',
        'column': 'KABC_PED',
        'mode': 'Pedestrian',
    },
    {
        'name': 'KABC Bicycle Score',
        'column': 'KABC_PDC',
        'mode': 'Bicycle',
    },
]

In [ ]:
# Define layer styling function

def hin_styling(feature):
    """
    Function to style the HIN layer based on the provided metric tier column.
    """
    # Get the tier from the feature properties
    tier = feature['properties']['__style__']

    # Define the color map for the KABC Vehicle Score
    color_map = {
        'Minimal':  '#ffffcc',
        'Low':      '#ffeda0',
        'Medium':   '#feb24c',
        'High':     '#f03b20',
        'Critical': '#bd0026',
    }
    weight_map = {
        'Minimal':  1.8,
        'Low':      1.8,
        'Medium':   2.5,
        'High':     3.5,
        'Critical': 5.5,
    }
    weight_map_bg = {
        'Minimal':  3.0,
        'Low':      3.0,
        'Medium':   4.5,
        'High':     5.5,
        'Critical': 7.5,
    }
    
    # Get styling parameters
    # - Style as background
    if feature['properties']['__bg__']:
        color = '#000000' # Black background
        weight = weight_map_bg.get(tier, 1)

    # - Style as foreground
    else:
        color = color_map.get(tier, '#000000')
        weight = weight_map.get(tier, 1)
    
    return {
        'fillColor': color,
        'color': color,
        'weight': weight,
        'fillOpacity': 1.0,
    }

In [ ]:
# Initialize the map with `folium`
m = folium.Map(tiles=None) # No tiles to start--we'll add these later

# Iterate over all HIN layers
hin_feature_groups = []
for PARAMS in HIN_PARAMS:
    # Identify the target metric
    metric = PARAMS['column']
    mode   = PARAMS['mode']
    print(f'Adding {mode} HIN segments to map ({metric})...')

    # Create a doubled version of the data, sorted for the target metric and with
    # an additional column to indicate background copies of each feature
    data = segments.copy().to_crs('EPSG:4326').sort_values(by=f'{metric}_SCORE')
    data['__style__'] = data[f'{metric}_SCORE_TIER']
    datas = pd.concat([data.assign(__bg__=True), data.assign(__bg__=False)], ignore_index=True)
    data = datas.to_json()

    # Customize map data popups
    popup = folium.GeoJsonPopup(
        fields=[f'{metric}_SCORE', f'{metric}_SCORE_PCT', f'{metric}_SCORE_TIER'],
        aliases=[f'{mode} Score', f'{mode} Percentile', f'{mode} Tier'],
        localize=False,
        labels=True,
        max_width=600
    )

    # Add HIN segments to the map
    feature_group = folium.GeoJson(
        data=data,
        name=f'{mode} HIN Segments',
        style_function=hin_styling,
        popup=popup,
    ).add_to(m)

    # Log feature group
    hin_feature_groups.append(feature_group)

In [ ]:
# Add basemap tile layers
folium.TileLayer('OpenStreetMap', name='Street Map', show=True).add_to(m)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    name='Esri Aerial Imagery', attr='Esri', show=False).add_to(m)
folium.TileLayer('cartodbpositron', name='Bright', show=False).add_to(m)
folium.TileLayer('cartodbdark_matter', name='Dark', show=False).add_to(m)

# Add a step colormap
branca.colormap.StepColormap(
    colors=['#ffffcc', '#ffeda0', '#feb24c', '#f03b20', '#bd0026'],
    index=[0.00, 0.50, 0.75, 0.85, 0.95],
    vmin=0,
    vmax=1,
    tick_labels=[],
    caption='High Injury Network Score (Minimal, Low, Medium, High, Critical)',
    
).add_to(m)

# Add layer controls
from folium.plugins import GroupedLayerControl
folium.map.LayerControl('bottomright', collapsed=True, autoZIndex=True, draggable=False).add_to(m)
GroupedLayerControl(
    groups={
        'Modal HINs': hin_feature_groups,
    },
    collapsed=False,
).add_to(m)

# Fit the map to the bounds of its features
m.fit_bounds(m.get_bounds())

# Export HTML file
fp = os.path.join('..', '99_Resources', 'hin-webmap-advanced.html')
m.save(fp)